In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.env import NearFieldBeamFocusingEnv
from src.sac_agent import SACAgent
from src.utils import load_hfss_channel
from src.config import (
    ENV_CONFIG,
    SAC_CONFIG,
    TRAIN_CONFIG,
    HFSS_FILE
)

from src.utils import (
    near_field_channel_to_user,
    near_field_conjugate_weights,
    sph_to_cart
)
from collections import deque
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
%pip install import-ipynb
import import_ipynb
import torch
import copy


[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: /usr/local/bin/python3.13 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from torch import nn

class Actor(nn.Module):
    def __init__(self, obs_dim, action_dim, max_action=1.0):

        self.input_features = obs_dim
        self.output_labels = 2 * action_dim
        super().__init__()
        # Initialise the Normal probability distribution variables. Or do we not need to do that ?
        self.action_dim = action_dim

        self.max_action = max_action
        self.actor_network = nn.Sequential(
            nn.Linear(self.input_features, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, self.output_labels),
        )

        # initialise the neural network parameters randomly ?

    def sample(self, state):
        mu, log_sigma = self.forward(state)
        log_sigma = log_sigma.clamp(-20, 2)
        sigma = log_sigma.exp()

        distribution = torch.distributions.Normal(mu, sigma)

        # Equivalent to mu + sigma * torch.randn_like(mu)
        u = distribution.rsample()

        squashed = torch.tanh(u)
        scale = torch.as_tensor(
            self.max_action, dtype=mu.dtype, device=mu.device
        )
        action = squashed * scale

        # Correct the density for tanh and action scaling.
        log_prob = distribution.log_prob(u)
        log_prob -= torch.log(scale) + 2 * (np.log(2.0) - u - F.softplus(-2 * u))
        log_prob = log_prob.sum(dim=-1, keepdim=True)

        return action, log_prob

    def forward(self, states):
        # transforms a state into Gaussian parameters σ and μ
        # fully connected DNN
        # in : state = ([x y ω]) (three scalars)
        # out: mean and log-standard-deviation vectors
        outputs = self.actor_network(states)
        return outputs.split(self.action_dim, dim=-1)


In [3]:
import random
class ReplayBuffer:
    def __init__(self, capacity=100_000):
        self.replay = deque(maxlen=capacity)

    def __len__(self):
        return len(self.replay)

    def record(self,state,action,reward,next_state,terminal=False):
        # stored at each increment.
        self.replay.append((
            np.array(state, copy=True),
            np.array(action, copy=True),
            float(reward),
            np.array(next_state, copy=True),
            bool(terminal),
        ))

    def sample(self, batch_size):
        batch = random.sample(list(self.replay), batch_size)
        states, actions, rewards, next_states, terminals = zip(*batch)

        return (
            torch.as_tensor(np.stack(states), dtype=torch.float32),
            torch.as_tensor(np.stack(actions), dtype=torch.float32),
            torch.as_tensor(rewards, dtype=torch.float32).reshape(-1, 1),
            torch.as_tensor(np.stack(next_states), dtype=torch.float32),
            torch.as_tensor(terminals, dtype=torch.float32).reshape(-1, 1),
        )


In [4]:
class Critic(nn.Module):
    def __init__(self, obs_dim, action_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(obs_dim + action_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
        )

        # initialise the neural network parameters randomly ?

    def forward(self, states, actions):
        inputs = torch.cat([states, actions], dim=-1)
        return self.network(inputs)


In [5]:
import random
class ReplayBuffer:
    def __init__(self, capacity=100_000):
        self.replay = deque(maxlen=capacity)

    def __len__(self):
        return len(self.replay)

    def record(self,state,action,reward,next_state,terminal=False):
        # stored at each increment.
        self.replay.append((
            np.array(state, copy=True),
            np.array(action, copy=True),
            float(reward),
            np.array(next_state, copy=True),
            bool(terminal),
        ))

    def sample(self, batch_size):
        batch = random.sample(list(self.replay), batch_size)
        states, actions, rewards, next_states, terminals = zip(*batch)

        return (
            torch.as_tensor(np.stack(states), dtype=torch.float32),
            torch.as_tensor(np.stack(actions), dtype=torch.float32),
            torch.as_tensor(rewards, dtype=torch.float32).reshape(-1, 1),
            torch.as_tensor(np.stack(next_states), dtype=torch.float32),
            torch.as_tensor(terminals, dtype=torch.float32).reshape(-1, 1),
        )


In [6]:

hsi, selected_freq = load_hfss_channel(HFSS_FILE, target_freq=ENV_CONFIG['fc'])

def make_env(seed):
    config = dict(ENV_CONFIG)
    config['seed'] = seed
    return NearFieldBeamFocusingEnv(hsi, **config)
SEED = 1234

env = make_env(SEED)
obs_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
assert np.all(env.action_space.low == -1) and np.all(env.action_space.high == 1)
print('Observation dimensions:', obs_dim)
print('Action dimensions:', action_dim)
print('Incremental actions:', not env.direct_action_mode)
print('Selected frequency:', selected_freq)

Observation dimensions: 19
Action dimensions: 6
Incremental actions: True
Selected frequency: 3500000000.0


In [7]:
import copy

q1 = Critic(obs_dim, action_dim)
q2 = Critic(obs_dim, action_dim)  # Separate, independently initialized network

q1_optimizer = torch.optim.Adam(
    q1.parameters(),
    lr=3e-4,
)

q2_optimizer = torch.optim.Adam(
    q2.parameters(),
    lr=3e-4,
)

target_q1 = copy.deepcopy(q1)
target_q2 = copy.deepcopy(q2)

target_q1.requires_grad_(False)
target_q2.requires_grad_(False)



buffer = ReplayBuffer()

actor = Actor(obs_dim, action_dim, max_action=1.0)
actor_optimizer = torch.optim.Adam(
    actor.parameters(),
    lr=3e-4,
)


In [8]:
mse_loss = nn.MSELoss()

def update_critics(states, actions, targets):
    # Predictions for actions stored in the replay buffer.
    q1_predictions = q1(states, actions)
    q2_predictions = q2(states, actions)

    # Compare predictions with the fixed learning targets.
    q1_loss = mse_loss(q1_predictions, targets.detach())
    q2_loss = mse_loss(q2_predictions, targets.detach())
    critic_loss = q1_loss + q2_loss

    # Clear old gradients, calculate new ones, then update weights.
    q1_optimizer.zero_grad()
    q2_optimizer.zero_grad()

    critic_loss.backward()

    q1_optimizer.step()
    q2_optimizer.step()

    return critic_loss.item()


In [9]:
def compute_actor_loss(actor, q1, q2, states, alpha=0.9):
    # Fresh actions sampled at the replay batch's current states.
    actions, log_prob = actor.sample(states)

    q_value = torch.minimum(
        q1(states, actions),
        q2(states, actions),
    )

    return (alpha * log_prob - q_value).mean()

def update_actor(states, alpha):
    # Freeze critic parameters while retaining gradients through their inputs.
    critic_parameters = list(q1.parameters()) + list(q2.parameters())
    for parameter in critic_parameters:
        parameter.requires_grad_(False)

    actor_optimizer.zero_grad()

    actor_loss = compute_actor_loss(actor, q1, q2, states, alpha=alpha)
    actor_loss.backward()
    actor_optimizer.step()

    for parameter in critic_parameters:
        parameter.requires_grad_(True)
    return actor_loss.item()


In [10]:
def compute_critic_targets(actor, target_q1, target_q2,next_states, rewards, terminals,gamma=0.9, alpha=0.9):
    # uses data from the replay buffer
    with torch.no_grad():
        next_actions, next_log_prob = actor.sample(next_states)

        next_q = torch.minimum(
            target_q1(next_states, next_actions),
            target_q2(next_states, next_actions),
        )

        targets = rewards + gamma * (1.0 - terminals.float()) * (
            next_q - alpha * next_log_prob
        )

    return targets

def update_target_networks(q1, q2, target_q1, target_q2, tau=0.005):
    with torch.no_grad():
        for critic, target_critic in [
            (q1, target_q1),
            (q2, target_q2),
        ]:
            for parameter, target_parameter in zip(
                critic.parameters(), target_critic.parameters()
            ):
                target_parameter.mul_(1 - tau)
                target_parameter.add_(tau * parameter)

#slowly copying weights


In [11]:
batch_size = 256
GAMMA = 0.99
ENTROPY_ALPHA = 0.2  # Starting value; its suitability must be checked for this reward scale.
TAU = 0.005

def update_agent():
    states, actions, rewards, next_states, terminals = buffer.sample(batch_size)
    targets = compute_critic_targets(
        actor, target_q1, target_q2, next_states, rewards, terminals,
        gamma=GAMMA, alpha=ENTROPY_ALPHA,
    )
    critic_loss = update_critics(states, actions, targets)
    actor_loss = update_actor(states, alpha=ENTROPY_ALPHA)
    update_target_networks(q1, q2, target_q1, target_q2, tau=TAU)
    return critic_loss, actor_loss


In [12]:
with torch.no_grad():
    test_states = torch.zeros(8, obs_dim)
    test_actions, test_logp = actor.sample(test_states)
    mu, log_std = actor(test_states)
    test_q = q1(test_states, test_actions)
assert mu.shape == log_std.shape == test_actions.shape == (8, action_dim)
assert test_logp.shape == test_q.shape == (8, 1)
assert torch.isfinite(test_logp).all()
assert (test_actions.abs() <= 1).all()
print('Network dimensions and finite outputs checked.')


Network dimensions and finite outputs checked.


In [18]:
def train_agent(env, total_steps=50_000, learning_starts=1_000):
    state, _ = env.reset(seed=SEED)
    env.action_space.seed(SEED)
    episode_return = 0.0
    history = []
    for step in range(total_steps):
        if step < learning_starts:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                action_t, _ = actor.sample(torch.as_tensor(state, dtype=torch.float32).unsqueeze(0))
            action = action_t.squeeze(0).numpy()
        next_state, reward, terminated, truncated, info = env.step(action)
        buffer.record(state, action, reward, next_state, terminated)
        episode_return += reward
        if step >= learning_starts and len(buffer) >= batch_size:
            losses = update_agent()
            if not np.isfinite(losses).all():
                raise FloatingPointError(f'Nonfinite losses at step {step}: {losses}')
        if terminated or truncated:
            si_improvement = info["SI_ref_dB"] - info["SI_dB"]
            dl_loss = info["DL_gain_loss_dB"]
            ul_loss = info["UL_gain_loss_dB"]
            history.append(dict(step=step + 1, episode_return=episode_return,
                                SI_dB=info['SI_dB'], DL_gain_loss_dB=info['DL_gain_loss_dB'],
                                UL_gain_loss_dB=info['UL_gain_loss_dB']))
            if len(history) % 50 == 0:
                print(
                    f"Step {step + 1}: "
                    f"return={episode_return:.2f}, "
                    f"SI improvement={si_improvement:+.2f} dB, "
                    f"DL gain loss={dl_loss:.2f} dB, "
                    f"UL gain loss={ul_loss:.2f} dB"
                )
            state, _ = env.reset()
            episode_return = 0.0
        else:
            state = next_state
    return history

# Run this cell deliberately when ready for training.
history = train_agent(env, total_steps=50_000)


Step 3000: return=193.72, SI improvement=-11.08 dB, DL gain loss=10.76 dB, UL gain loss=1.33 dB
Step 6000: return=-765.00, SI improvement=-17.91 dB, DL gain loss=10.76 dB, UL gain loss=11.31 dB
Step 9000: return=-546.13, SI improvement=-0.48 dB, DL gain loss=0.32 dB, UL gain loss=11.19 dB
Step 12000: return=760.92, SI improvement=+7.75 dB, DL gain loss=5.11 dB, UL gain loss=1.00 dB
Step 15000: return=-837.73, SI improvement=-27.65 dB, DL gain loss=10.12 dB, UL gain loss=12.75 dB
Step 18000: return=-284.81, SI improvement=+6.10 dB, DL gain loss=3.39 dB, UL gain loss=1.09 dB
Step 21000: return=-132.84, SI improvement=-10.68 dB, DL gain loss=12.29 dB, UL gain loss=0.88 dB
Step 24000: return=-592.48, SI improvement=-2.35 dB, DL gain loss=2.79 dB, UL gain loss=1.91 dB
Step 27000: return=-502.27, SI improvement=-17.52 dB, DL gain loss=9.84 dB, UL gain loss=5.95 dB
Step 30000: return=495.54, SI improvement=+0.50 dB, DL gain loss=1.86 dB, UL gain loss=1.95 dB
Step 33000: return=-318.33, SI imp

In [17]:
def evaluate_actor(actor, episodes=20, seed=20_000):
    evaluation_env = make_env(seed)
    rows = []
    was_training = actor.training
    actor.eval()
    try:
        for episode in range(episodes):
            state, _ = evaluation_env.reset(seed=seed + episode)
            total_reward = 0.0
            while True:
                with torch.no_grad():
                    mu, _ = actor(torch.as_tensor(state, dtype=torch.float32).unsqueeze(0))
                    action = (actor.max_action * torch.tanh(mu)).squeeze(0).numpy()
                state, reward, terminated, truncated, info = evaluation_env.step(action)
                total_reward += reward
                if terminated or truncated:
                    break
            feasible = (info['DL_gain_loss_dB'] <= evaluation_env.dl_gain_margin_db
                        and info['UL_gain_loss_dB'] <= evaluation_env.ul_gain_margin_db)
            rows.append(dict(episode_return=total_reward, SI_dB=info['SI_dB'],
                             SI_improvement_dB=info['SI_ref_dB'] - info['SI_dB'],
                             DL_gain_loss_dB=info['DL_gain_loss_dB'],
                             UL_gain_loss_dB=info['UL_gain_loss_dB'], feasible=feasible))
    finally:
        evaluation_env.close()
        actor.train(was_training)
    for key in ['episode_return', 'SI_dB', 'SI_improvement_dB', 'DL_gain_loss_dB', 'UL_gain_loss_dB']:
        values = np.array([row[key] for row in rows])
        print(f'{key}: mean={values.mean():.3f}, std={values.std():.3f}')
    print(f"Final-step feasible: {np.mean([row['feasible'] for row in rows]):.1%}")
    return rows

# Compare before and after training with the same evaluation seeds:
#before = evaluate_actor(actor)
#history = train_agent(env)
after = evaluate_actor(actor)


episode_return: mean=-141.431, std=418.897
SI_dB: mean=-27.692, std=12.072
SI_improvement_dB: mean=-1.730, std=10.077
DL_gain_loss_dB: mean=8.868, std=6.050
UL_gain_loss_dB: mean=2.649, std=1.969
Final-step feasible: 15.0%


In [15]:
%pip install torchview
from torchview import draw_graph
from IPython.display import display

actor_graph = draw_graph(
    actor,
    input_size=(1, 19),
    expand_nested=True,
)

critic_graph = draw_graph(
    q1,
    input_data=(
        torch.zeros(1, 19),  # State
        torch.zeros(1, 6),   # Action
    ),
    expand_nested=True,
)

#display(critic_graph.visual_graph)
#display(actor_graph.visual_graph)




[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: /usr/local/bin/python3.13 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
#train_agent(env)
print(env.dl_gain_margin_db) # 3.0dB
print(env.ul_gain_margin_db) # 3.0dB

3.0
3.0
